# Resultados RQ1–RQ3 — rodada final
Anotação individual e consolidada (RQ1), comparação pareada por fold do fine-tuning contra o benchmark, com e sem biO-IS (RQ1/RQ2),
composição da filtragem (RQ2) e restrição do espaço de rótulos em 2 fases (RQ3, books e dblp).
Saída: `data/results/mq/resultados_rq1_rq3/`.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "data" / "results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))

from src.config.datasets_collected import LABEL_MEANINGS
from src.systems.fine_tune_system.evaluation.paired_comparator import PairedSystemComparator
from src.systems.fine_tune_system.evaluation.supervision_comparison import SupervisionComparison
from src.systems.class_filter_system.validation.confusability import CandidateRebuilder, ConfusabilityAnalyzer
from loguru import logger
logger.remove(); logger.add(sys.stderr, level="WARNING")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "data" / "results"
OUT = RESULTS / "mq" / "resultados_rq1_rq3"
OUT.mkdir(parents=True, exist_ok=True)
XLSX = RESULTS / "mq" / "Roberta - Catizani.xlsx"

DATES = {
    "movie_review": "2026-09-24_12-48-42",
    "agnews": "2026-10-08_02-20-35",
    "books": "2026-09-13_07-19-54",
    "dblp": "2026-09-13_18-46-30",
}
TWO_PHASE = {"books": "2026-09-18_17-03-05", "dblp": "2026-09-18_17-03-44"}
MODELS = ["deepseek-r1-8b", "qwen3-8b", "llama3.1-8b"]
INVALID = -1
REPORT = {}


def coerce(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.replace({"ERROR": INVALID, "": INVALID, "N/A": INVALID}), errors="coerce").fillna(INVALID).astype(int)


def votes_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Votos dos 3 modelos, padrão de concordância (3x0/2x1/1x1x1) e rótulo majoritário (NaN se 1x1x1)"""
    v = pd.DataFrame({m: coerce(df[f"{m}_consensus"]) for m in MODELS}, index=df.index)
    n_unique = v.nunique(axis=1)
    pattern = n_unique.map({1: "3x0", 2: "2x1", 3: "1x1x1"})
    majority = v.mode(axis=1)[0].where(n_unique < 3)
    return v.assign(pattern=pattern, majority=majority)


def fleiss_kappa(votes: pd.DataFrame) -> float:
    """κ de Fleiss com -1 como categoria própria (cada texto tem exatamente 3 votos)"""
    cats = sorted(pd.unique(votes.values.ravel()))
    counts = np.stack([(votes.values == c).sum(axis=1) for c in cats], axis=1)
    n = counts.sum(axis=1)[0]
    p_j = counts.sum(axis=0) / counts.sum()
    p_i = ((counts ** 2).sum(axis=1) - n) / (n * (n - 1))
    p_bar, p_e = p_i.mean(), (p_j ** 2).sum()
    return (p_bar - p_e) / (1 - p_e)


def acc_f1(y, p):
    return accuracy_score(y, p), f1_score(y, p, average="macro")

## RQ1 — anotação individual e consolidada

In [2]:
per_model, consensus, strata = [], [], []
annotations = {}
for ds, date in DATES.items():
    run = RESULTS / ds / date
    df = pd.read_csv(run / "annotations.csv")
    df = df[df["ground_truth"].notna()].reset_index(drop=True)
    y = df["ground_truth"].astype(int)
    v = votes_frame(df)
    annotations[ds] = df.assign(**{f"vote_{m}": v[m] for m in MODELS}, pattern=v["pattern"], majority=v["majority"])

    for m in MODELS:
        ok = v[m] != INVALID
        a, f = acc_f1(y[ok], v[m][ok])
        per_model.append({"dataset": ds, "modelo": m, "acuracia": a, "f1_macro": f, "taxa_invalida": 1 - ok.mean(), "n_invalidas": int((~ok).sum())})

    summary = json.loads((run / "summary" / "sumario_experimento.json").read_text(encoding="utf-8"))
    report = summary["validation"]["classification_report"]
    kept = v["majority"].notna() & (v["majority"] != INVALID)
    a, f = acc_f1(y[kept], v["majority"][kept].astype(int))
    any_invalid = (v[MODELS] == INVALID).any(axis=1)
    cohen = pd.read_csv(run / "consensus" / "cohens_kappa.csv")
    consensus.append({
        "dataset": ds, "n_textos": len(df),
        "n_3x0": int((v["pattern"] == "3x0").sum()), "n_2x1": int((v["pattern"] == "2x1").sum()), "n_1x1x1": int((v["pattern"] == "1x1x1").sum()),
        "n_maioria_invalida": int((v["majority"] == INVALID).sum()),
        "n_com_voto_invalido": int(any_invalid.sum()),
        "n_treinavel": int((kept & ~any_invalid).sum()),
        "acuracia": a, "f1_macro": f,
        "acuracia_sumario": report["accuracy"], "f1_macro_sumario": report["macro avg"]["f1-score"],
        "fleiss": fleiss_kappa(v[MODELS]), "fleiss_sumario": summary["metrics"]["fleiss_kappa"],
        **{f"cohen_{r.annotator_1.split('_')[0]}__{r.annotator_2.split('_')[0]}": r.cohens_kappa for r in cohen.itertuples()},
    })
    for pat in ["3x0", "2x1"]:
        sel = kept & (v["pattern"] == pat)
        a, f = acc_f1(y[sel], v["majority"][sel].astype(int))
        strata.append({"dataset": ds, "padrao": pat, "n": int(sel.sum()), "share": sel.mean(), "acuracia": a, "f1_macro": f})

per_model, consensus, strata = map(pd.DataFrame, [per_model, consensus, strata])
best = per_model.loc[per_model.groupby("dataset")["f1_macro"].idxmax(), ["dataset", "modelo", "f1_macro"]].rename(columns={"modelo": "melhor_modelo", "f1_macro": "f1_melhor"})
consensus = consensus.merge(best, on="dataset").assign(delta_vs_melhor=lambda d: d["f1_macro"] - d["f1_melhor"])
print(per_model.round(4)); print(consensus.round(4).T); print(strata.round(4))
REPORT.update(rq1_por_modelo=per_model, rq1_consenso=consensus, rq1_estratos=strata)

         dataset          modelo  acuracia  f1_macro  taxa_invalida  n_invalidas
0   movie_review  deepseek-r1-8b    0.8946    0.8945         0.0100          107
1   movie_review        qwen3-8b    0.9151    0.9151         0.0000            0
2   movie_review     llama3.1-8b    0.8960    0.8960         0.0001            1
3         agnews  deepseek-r1-8b    0.8519    0.8487         0.0022          283
4         agnews        qwen3-8b    0.8662    0.8643         0.0000            0
5         agnews     llama3.1-8b    0.8648    0.8634         0.0000            0
6          books  deepseek-r1-8b    0.6190    0.6396         0.0062          209
7          books        qwen3-8b    0.7197    0.7333         0.0000            0
8          books     llama3.1-8b    0.7157    0.7257         0.0001            2
9           dblp  deepseek-r1-8b    0.6119    0.5830         0.0042          161
10          dblp        qwen3-8b    0.6330    0.6002         0.0000            0
11          dblp     llama3.

## RQ1/RQ2 — fine-tuning pareado por fold (benchmark × consolidado × consolidado + biO-IS)

In [3]:
comparison = SupervisionComparison(RESULTS, DATES, benchmark_xlsx=XLSX)
tables = comparison.run()
comparison.export(tables, RESULTS / "mq" / "comparacao_supervisao")

# IC da diferença ajustado por Bonferroni (nível 1 − α/m), coerente com o p corrigido
folds = tables["folds"]
paired = tables["paired"].copy()
bonf_lo, bonf_hi = [], []
for r in paired.itertuples():
    m = (paired["family"] == r.family).sum() // paired.loc[paired["family"] == r.family, "metric"].nunique()
    wide = folds[(folds["dataset"] == r.dataset) & (folds["metric"] == r.metric)].pivot_table(index="fold", columns="system", values="value")
    diff = (wide[r.system_a] - wide[r.system_b]).dropna()
    _, lo, hi = PairedSystemComparator(conf=1 - 0.05 / m).mean_ci(diff)
    bonf_lo.append(lo); bonf_hi.append(hi)
paired["diff_ci_bonf_low"], paired["diff_ci_bonf_high"] = bonf_lo, bonf_hi
print(tables["individual"].round(4)); print(paired.drop(columns=["n_folds"]).round(5))
REPORT.update(ft_individual=tables["individual"], ft_pareado=paired, ft_folds=folds)

# Referência não pareada: benchmark humano com e sem biO-IS (Cunha et al., só média e IC)
summary_sheet = pd.read_excel(XLSX, sheet_name="summary-Effect-Roberta", header=1).iloc[:, 1:]
human_is = summary_sheet[summary_sheet["dataset"].isin(["DBLP", "Books", "MR", "AGNews"])][["dataset", "NoSel", "biO-IS"]]
print(human_is)
REPORT["benchmark_humano_bio_is"] = human_is

         dataset    metric             system  n_folds    mean  ci_low  ci_high
0         agnews  accuracy         aggregated        5  0.8733  0.8715   0.8750
1         agnews  accuracy  aggregated_bio-is        5  0.8780  0.8740   0.8821
2         agnews  f1_macro         aggregated        5  0.8719  0.8702   0.8736
3         agnews  f1_macro  aggregated_bio-is        5  0.8773  0.8731   0.8815
4         agnews  f1_macro          benchmark        5  0.9173  0.9150   0.9196
5          books  accuracy         aggregated       10  0.7112  0.7039   0.7184
6          books  accuracy  aggregated_bio-is       10  0.7082  0.7028   0.7137
7          books  f1_macro         aggregated       10  0.7231  0.7162   0.7301
8          books  f1_macro  aggregated_bio-is       10  0.7209  0.7160   0.7259
9          books  f1_macro          benchmark       10  0.8721  0.8663   0.8778
10          dblp  accuracy         aggregated       10  0.6379  0.6354   0.6403
11          dblp  accuracy  aggregated_b

## RQ2 — o que a biO-IS remove

In [4]:
is_rows, is_strata = [], []
for ds, date in DATES.items():
    is_dir = RESULTS / ds / date / "instance_selection"
    rep = json.loads((is_dir / "instance_selection_report.json").read_text(encoding="utf-8"))
    removed = pd.read_csv(is_dir / "instancias_removidas.csv")
    kept = pd.read_csv(is_dir / "dataset_filtrado.csv")
    for d in (removed, kept):
        d["wrong"] = d["resolved_annotation"].astype(int) != d["ground_truth"].astype(int)
    total_wrong = removed["wrong"].sum() + kept["wrong"].sum()
    noise, red = removed[removed["removal_reason"] == "noise"], removed[removed["removal_reason"] == "redundant"]
    is_rows.append({
        "dataset": ds, "entrada": rep["labeled_instances"], "mantidas": rep["kept_instances"], "removidas": rep["removed_instances"],
        "frac_removida": rep["reduction_rate"], "ruidosas": len(noise), "redundantes": len(red),
        "razao_red_ruido": len(red) / len(noise),
        "erro_ruidosas": noise["wrong"].mean(), "erro_redundantes": red["wrong"].mean(),
        "erro_removidas": removed["wrong"].mean(), "erro_mantidas": kept["wrong"].mean(), "erro_entrada": total_wrong / (len(removed) + len(kept)),
        "frac_erros_capturados": removed["wrong"].sum() / total_wrong,
        "frac_removidas_corretas": 1 - removed["wrong"].mean(),
        "frac_removidas_3x0": (removed["consensus_level"] == "high").mean(),
        "frac_ruidosas_3x0": (noise["consensus_level"] == "high").mean(),
        "frac_redundantes_3x0": (red["consensus_level"] == "high").mean(),
        "weak_clf_acc": rep["weak_classifier_accuracy"], "weak_clf_f1": rep["weak_classifier_f1_macro"],
    })
    for (reason, level), g in removed.groupby(["removal_reason", "consensus_level"]):
        is_strata.append({"dataset": ds, "tipo": reason, "padrao": {"high": "3x0", "medium": "2x1"}.get(level, level), "n": len(g), "erro": g["wrong"].mean()})
is_rows, is_strata = pd.DataFrame(is_rows), pd.DataFrame(is_strata)
print(is_rows.round(4).T); print(is_strata.round(4))
REPORT.update(rq2_filtragem=is_rows, rq2_filtragem_estratos=is_strata)

                                    0       1       2       3
dataset                  movie_review  agnews   books    dblp
entrada                         10639  127261   32785   36999
mantidas                         6686   91925   21765   24933
removidas                        3953   35336   11020   12066
frac_removida                  0.3716  0.2777  0.3361  0.3261
ruidosas                         1294    3521    2824    2817
redundantes                      2659   31815    8196    9249
razao_red_ruido                2.0549  9.0358  2.9023  3.2833
erro_ruidosas                  0.1878  0.5257  0.4908  0.6471
erro_redundantes               0.0504  0.0984  0.2219  0.2659
erro_removidas                 0.0954   0.141  0.2908  0.3549
erro_mantidas                  0.0817  0.1265  0.2769  0.3509
erro_entrada                   0.0868  0.1305  0.2816  0.3522
frac_erros_capturados          0.4085  0.2998  0.3472  0.3286
frac_removidas_corretas        0.9046   0.859  0.7092  0.6451
frac_rem

## RQ3 — restrição do espaço de rótulos (books e dblp)

In [5]:
recall_rows, cmp_rows, cond_rows, per_class_rows, decomposition = [], [], [], [], []
confusions, entropies, pairs_tables, overlap = {}, {}, {}, {}
SYSTEMS = MODELS + ["consenso"]

for ds, ts in TWO_PHASE.items():
    run = RESULTS / ds / "two_phase" / ts
    names = {int(k): v for k, v in LABEL_MEANINGS[ds].items()}
    agg = pd.read_csv(run / "recall_at_k_aggregated.csv")
    for r in agg.itertuples():
        recall_rows.append({"dataset": ds, "k": r.k, "recall_macro": r.recall_at_k_macro_mean, "recall_macro_std": r.recall_at_k_macro_std,
                            "recall_micro": r.recall_at_k_mean, "recall_micro_std": r.recall_at_k_std})
    saved_recall = pd.read_csv(run / "recall_at_k_all_folds.csv").query("k == 3").set_index("fold")["recall_at_k"]

    base = annotations[ds].set_index("text_id")
    frames = []
    for cand in CandidateRebuilder(ds, cache_dir=str(ROOT / "data" / ".cache")).folds():
        fold = int(cand["fold"].iloc[0])
        hits = [y in c for y, c in zip(cand["label"], cand["candidates"])]
        assert abs(np.mean(hits) - saved_recall[fold]) < 1e-9, f"{ds} fold {fold}: candidatas não reproduzem o recall salvo"
        filt = pd.read_csv(run / f"fold_{fold}" / "filtered" / "annotations.csv")
        fv = votes_frame(filt)
        filt = pd.DataFrame({"text_id": filt["text_id"], **{f"r_{m}": fv[m] for m in MODELS}, "r_consenso": fv["majority"].fillna(-2).astype(int),
                             "gt_r": filt["ground_truth"].astype(int)})
        b = base.loc[base.index.intersection(filt["text_id"])]
        bl = pd.DataFrame({"text_id": b.index, **{f"u_{m}": b[f"vote_{m}"].values for m in MODELS},
                           "u_consenso": b["majority"].fillna(-2).astype(int).values, "gt_u": b["ground_truth"].astype(int).values})
        frames.append(cand.merge(filt, on="text_id").merge(bl, on="text_id"))
    data = pd.concat(frames, ignore_index=True)
    assert (data["gt_r"] == data["label"]).all() and (data["gt_u"] == data["label"]).all()
    overlap[ds] = {"linhas": len(data), "textos_unicos": data["text_id"].nunique()}

    # Pareado por texto dentro do fold; inferência entre folds
    for s in SYSTEMS:
        for fold, g in data.groupby("fold"):
            ok = g[f"r_{s}"].ge(0) & g[f"u_{s}"].ge(0)
            ar, fr = acc_f1(g.loc[ok, "label"], g.loc[ok, f"r_{s}"])
            au, fu = acc_f1(g.loc[ok, "label"], g.loc[ok, f"u_{s}"])
            cmp_rows.append({"dataset": ds, "sistema": s, "fold": fold, "n_pareado": int(ok.sum()), "n_holdout": len(g),
                             "f1_restrito": fr, "f1_sem_restricao": fu, "acc_restrito": ar, "acc_sem_restricao": au,
                             "inval_restrito": (g[f"r_{s}"] < 0).mean(), "inval_sem_restricao": (g[f"u_{s}"] < 0).mean()})

    # Erro condicional y ∈ Top-3, nas duas condições, sobre os mesmos textos válidos
    analyzer = ConfusabilityAnalyzer(names)
    in_top = np.array([y in c for y, c in zip(data["label"], data["candidates"])])
    for s in SYSTEMS:
        ok = data[f"r_{s}"].ge(0) & data[f"u_{s}"].ge(0)
        d = data[ok]
        it = in_top[ok.values]
        err_r, err_u = d[f"r_{s}"] != d["label"], d[f"u_{s}"] != d["label"]
        cond_rows.append({"dataset": ds, "sistema": s, "n_validos": len(d), "frac_y_in_top3": it.mean(),
                          "p_err_restrito_dado_in": err_r[it].mean(), "p_err_sem_restricao_dado_in": err_u[it].mean(),
                          "p_err_sem_restricao_dado_out": err_u[~it].mean(),
                          "erros_restrito_total": int(err_r.sum()), "frac_erros_restrito_y_fora_top3": (err_r & ~it).sum() / err_r.sum()})
    # Consenso: mesmos textos válidos nas duas condições (sem 1x1x1 e sem maioria inválida)
    data_valid = data[data["r_consenso"].ge(0) & data["u_consenso"].ge(0)]
    ce_r = analyzer.conditional_error(data_valid, "r_consenso").rename(columns={"p_error": "p_err_restrito", "n": "n_in"})
    ce_u = analyzer.conditional_error(data_valid, "u_consenso")[["label", "p_error"]].rename(columns={"p_error": "p_err_sem_restricao"})
    recall_class = pd.DataFrame({"label": data["label"], "hit": in_top}).groupby("label")["hit"].mean().rename("recall_at_3").reset_index()
    per_class_rows.append(ce_r.merge(ce_u, on="label").merge(recall_class, on="label", how="left").assign(dataset=ds))
    confusions[ds] = analyzer.conditional_confusion(data_valid, "r_consenso")
    entropies[ds] = analyzer.error_entropy(data_valid, "r_consenso").assign(dataset=ds)
    pairs_tables[ds] = analyzer.top_pairs(data_valid, "r_consenso", n=8).assign(dataset=ds)
    REPORT[f"rq3_confusao_{ds}"] = confusions[ds].reset_index().rename(columns={"index": "verdadeira"})

recall_rows, cmp_folds, cond_rows = pd.DataFrame(recall_rows), pd.DataFrame(cmp_rows), pd.DataFrame(cond_rows)
per_class = pd.concat(per_class_rows, ignore_index=True)

# Inferência pareada entre folds (Δ = restrito − sem restrição), família = 2 datasets × 4 sistemas por métrica
cmp_rows = []
pc = PairedSystemComparator()
for metric in ["f1", "acc"]:
    for (ds, s), g in cmp_folds.groupby(["dataset", "sistema"]):
        diff = g[f"{metric}_restrito"] - g[f"{metric}_sem_restricao"]
        m, lo, hi = pc.mean_ci(diff)
        _, blo, bhi = PairedSystemComparator(conf=1 - 0.05 / 8).mean_ci(diff)
        cmp_rows.append({"metrica": metric, "dataset": ds, "sistema": s, "n_folds": len(g), "n_pareado_medio": g["n_pareado"].mean(),
                         "restrito": g[f"{metric}_restrito"].mean(), "sem_restricao": g[f"{metric}_sem_restricao"].mean(),
                         "delta": m, "ic_low": lo, "ic_high": hi, "ic_bonf_low": blo, "ic_bonf_high": bhi,
                         "p": stats.ttest_rel(g[f"{metric}_restrito"], g[f"{metric}_sem_restricao"]).pvalue,
                         "inval_restrito": g["inval_restrito"].mean(), "inval_sem_restricao": g["inval_sem_restricao"].mean()})
rq3_cmp = pd.DataFrame(cmp_rows)
for metric in ["f1", "acc"]:
    sel = rq3_cmp["metrica"] == metric
    rq3_cmp.loc[sel, "p_bonferroni"] = multipletests(rq3_cmp.loc[sel, "p"], method="bonferroni")[1]
    rq3_cmp.loc[sel, "p_bh"] = multipletests(rq3_cmp.loc[sel, "p"], method="fdr_bh")[1]

print(recall_rows.round(4)); print(rq3_cmp.round(5)); print(cond_rows.round(4)); print(per_class.round(4))
for ds in TWO_PHASE:
    print(confusions[ds].round(2)); print(entropies[ds].round(3)); print(pairs_tables[ds].round(3))
print(overlap)
REPORT.update(rq3_recall=recall_rows, rq3_pareado=rq3_cmp, rq3_pareado_folds=cmp_folds, rq3_condicional=cond_rows,
              rq3_por_classe=per_class, rq3_entropia=pd.concat(entropies.values()), rq3_pares=pd.concat(pairs_tables.values()),
              rq3_sobreposicao=pd.DataFrame(overlap).T.reset_index())

2026-10-14 19:44:41.413 | ERROR    | src.utils.data_loader:load_hf_dataset:183 - Erro ao carregar dataset: 404 Client Error. (Request ID: Root=1-6ad005d9-4f23abd659655c1a0049baa8;bddb2f34-2801-4611-b602-c26d4a3c0fd6)

Entry Not Found for url: https://huggingface.co/datasets/waashk/books/resolve/main/train_fold_10.parquet.


2026-10-14 19:45:56.867 | ERROR    | src.utils.data_loader:load_hf_dataset:183 - Erro ao carregar dataset: 404 Client Error. (Request ID: Root=1-6ad00624-324ca6442b0710fa53c73a4a;7da419bb-fbd2-4faa-8bd9-5e38ce806bb3)

Entry Not Found for url: https://huggingface.co/datasets/waashk/dblp/resolve/main/train_fold_10.parquet.


  dataset  k  recall_macro  recall_macro_std  recall_micro  recall_micro_std
0   books  2        0.9235            0.0028        0.9315            0.0021
1   books  3        0.9617            0.0017        0.9695            0.0012
2   books  4        0.9775            0.0026        0.9851            0.0009
3    dblp  2        0.9028            0.0041        0.9288            0.0018
4    dblp  3        0.9522            0.0034        0.9678            0.0018
5    dblp  4        0.9736            0.0027        0.9833            0.0013
   metrica dataset         sistema  n_folds  n_pareado_medio  restrito  sem_restricao    delta   ic_low  ic_high  ic_bonf_low  ic_bonf_high        p  inval_restrito  inval_sem_restricao  p_bonferroni     p_bh
0       f1   books        consenso       10           5790.9   0.74575        0.73279  0.01295  0.01164  0.01426      0.01090       0.01501  0.00000         0.02029              0.02461       0.00000  0.00000
1       f1   books  deepseek-r1-8b       10

## Exportar

In [6]:
payload = {
    "descricao": "Resultados RQ1–RQ3 da rodada final (anotação, consenso, fine-tuning pareado por fold, biO-IS e 2 fases)",
    "anotacoes": DATES, "two_phase": TWO_PHASE,
    **{k: json.loads(v.to_json(orient="records", force_ascii=False)) for k, v in REPORT.items()},
}
(OUT / "resultados_rq1_rq3.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
for k, v in REPORT.items():
    v.to_csv(OUT / f"{k}.csv", index=False, encoding="utf-8")
print("salvo em", OUT)

salvo em C:\Users\gabri\Documents\GitHub\llm-annotation\data\results\mq\resultados_rq1_rq3
